# SINDYc 2 · A world model with control (exp06, exploratory)

Notebook 1 collected battles where Blue's **intent** $u$ (advance / hold / withdraw, nearest /
focus) switches every 1-4 s. Here a model $\dot x = f(x, u)$ is identified from them and tested.

1. Data and derivatives
2. The reach closure, now with formation width
3. The candidate library and the SINDYc fit
4. Sparsity against accuracy (threshold sweep)
5. Predicting ahead, against baselines and a generic polynomial SINDYc
6. Which part limits accuracy? (oracle inputs)
7. Are the terms stable? (bootstrap by battle)
8. What is each control term worth? (ablations)
9. How much data does it need?
10. Does the derivative method matter?
11. Formation shape as a state

The model lives in `sindy_wm.models.intent_model`. Scale: 1 = ten full-health marines; time in
seconds; all errors are mean absolute errors on **validation battles** (new seeds), with the
real intents applied over the horizon.

In [ ]:
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps
from scipy.optimize import curve_fit

from sindy_wm.models.intent_model import (
    ALIVE_CONTROL_TERMS,
    CTRL,
    DT,
    SHAPE,
    STATE,
    IntentModel,
    add_controls,
    add_rates,
    fit_closures,
    horizon_errors,
    make_batch,
    r2,
)
from sindy_wm.paths import DATA_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
PROCESSED = DATA_DIR / "processed" / "exp06_intents"

## 1 · Data and derivatives

6,000 training battles and 1,000 validation battles, each with intents switching every 1-4 s.
`add_rates` adds smoothed derivatives (Savitzky-Golay, window 9, as in the earlier notebooks).
Rows where one side is wiped out are dropped for fitting.

In [ ]:
train = add_rates(add_controls(pd.read_parquet(PROCESSED / "train.parquet")))
val = add_rates(add_controls(pd.read_parquet(PROCESSED / "val.parquet")))
rows = train[(train["alive_blue"] > 0) & (train["alive_red"] > 0)]
val_rows = val[(val["alive_blue"] > 0) & (val["alive_red"] > 0)]
B = make_batch(val)  # start points every 1 s, with the real intents and the truth 8 s ahead
print(
    f"{train.index.get_level_values(0).nunique()} training battles, {len(rows):,} rows; "
    f"{len(B['X0']):,} validation start points"
)
rows.groupby(["movement", "target"]).size().unstack().assign(share=lambda t: t.sum(axis=1) / len(rows)).round(
    2
)

## 2 · The reach closure, now with formation width

Health losses follow the opponent's **units within reach** (`engagement_from_distance.ipynb`,
section 9). Reach is computed from the team geometry: reach = alive x share. With intents the
formations stretch and bend more than in exp05, and the old closure (distance and depth only)
fits Red badly. Three forms, fitted on training rows, scored on validation rows:

- **section 11:** a logistic step in distance, shifted by the enemy's depth, widened by own depth
- **the same as a normal CDF**
- **with width:** the distance replaced by $\sqrt{d^2 + k\,(w_{own} - w_{opp})^2}$: when the two
  formations differ in width, flank units are further from the enemy than the centroids say

In [ ]:
from scipy.special import expit, ndtr


def inputs(a, side):
    opp = "red" if side == "blue" else "blue"
    return tuple(
        a[c].to_numpy()
        for c in ["distance", f"depth_{side}", f"depth_{opp}", f"width_{side}", f"width_{opp}"]
    )


forms = {
    "logistic (section 11)": (
        lambda X, c, bo, bp, w0, a: expit(
            -(X[0] - c - bo * X[1] - bp * X[2]) / np.maximum(w0 + a * X[1], 0.01)
        ),
        (6, 0, 1, 0.3, 0.5),
    ),
    "normal CDF": (
        lambda X, c, bo, bp, w0, a: ndtr(
            (c + bo * X[1] + bp * X[2] - X[0]) / np.sqrt(w0**2 + (a * X[1]) ** 2 + 1e-4)
        ),
        (6, 0, 1, 0.3, 1),
    ),
    "normal CDF + width": (
        lambda X, c, bo, bp, w0, a, k: ndtr(
            (c + bo * X[1] + bp * X[2] - np.sqrt(X[0] ** 2 + k * (X[3] - X[4]) ** 2))
            / np.sqrt(w0**2 + (a * X[1]) ** 2 + 1e-4)
        ),
        (6, 0, 1, 0.3, 1, 1),
    ),
}
sample = rows.sample(200_000, random_state=0)
closure_table = {}
for side in ["blue", "red"]:
    for name, (f, p0) in forms.items():
        p, _ = curve_fit(
            f, inputs(sample, side), sample[f"reach_{side}"] / sample[f"alive_{side}"], p0=p0, maxfev=60000
        )
        closure_table[(side, name)] = {
            "R² (validation)": r2(
                val_rows[f"reach_{side}"], val_rows[f"alive_{side}"] * f(inputs(val_rows, side), *p)
            ),
            "switch c": p[0],
            **{
                m: r2(g[f"reach_{side}"], g[f"alive_{side}"] * f(inputs(g, side), *p))
                for m, g in val_rows.groupby("movement")
            },
        }
pd.DataFrame(closure_table).T.round(3)

**What the numbers say:** width matters, mostly for Red (R² 0.70 → 0.82) and for Blue (0.87 →
0.92). The switch still lands near the per-unit reach of 5.75. The weak spot is **Red's reach
while Blue withdraws** (about 0.6): Red is chasing, and the team geometry says little about who
has caught up with whom.

The closures (reach share per side, and the movement gates of section 3) are fitted once:

In [ ]:
closure_file = PROCESSED / "closures.pkl"
if closure_file.exists():
    P = pickle.load(open(closure_file, "rb"))
else:
    P = fit_closures(rows)
    pickle.dump(P, open(closure_file, "wb"))
pd.DataFrame(
    {side: P[f"share_{side}"] for side in ["blue", "red"]}, index=["c", "b_own", "b_opp", "w0", "a", "k"]
).T.round(2)

## 3 · The candidate library and the SINDYc fit

SINDYc fits $\dot x = \Theta(x, u)\,\xi$ with a sparse $\xi$. Notebook 1 showed that the control
effects are **state-dependent** (withdrawing only matters for units in reach; advancing only
moves units that are out of contact), so the library holds *products* of state terms and controls:

| equation | candidates |
|---|---|
| health (both sides) | {`Rb`, `Rr` units in reach, `Ab`, `Ar` alive} x {1, hold, wd, focus}: 16 terms |
| distance | closing gate per movement `close*adv`, `close*hold`, `close*wd`, and `flee*wd` |
| alive | $(\log A)' = p(u)\,(\log H)'$ with $p$ = {1, hold, wd, focus}: units die as health is lost |

The gates' shapes (where closing stops) are nonlinear and fitted by `curve_fit`
(`fit_movement_gates`); SINDy chooses the terms and their coefficients. STLSQ runs **per
equation on standardised features**, so the threshold is an effect size and each equation gets
its own. The health equations can be fitted on the closure's reach or on the **measured reach**
(cleaner firepower coefficients; the closure is always used when predicting).

In [ ]:
print("threshold 0.05, fitted on the closure's reach")
rich = IntentModel(P).fit(rows, thr_health=0.05, verbose=True, thr_shape=None)
print("\nthreshold 0.2, fitted on the measured reach")
sparse = IntentModel(P).fit(rows, thr_health=0.2, reach_true=True, verbose=True, thr_shape=None)

**What the numbers say.** At the higher threshold, on measured reach, the model is:

- `(health_blue)' = -0.176 Rr`: Blue loses 0.176 per Red unit in reach. **No control term
  survives:** Red's fire doesn't depend on Blue's intent.
- `(health_red)' = -0.185 Rb + 0.164 Rb*wd`: Red loses 0.185 per Blue unit in reach, and
  **withdrawing removes 89% of it** (0.164 / 0.185).
- `(distance)' = -6.0 close*adv - 3.2 close*hold - 3.2 close*wd + 0.3 flee*wd`: closing at about 6
  when both sides walk, 3.2 when only Red does: a marine's speed is 3.15.
- `(log alive_red)' = [0.49 + 0.15 focus] (log health_red)'`: **focus fire kills units sooner**
  for the same health lost. Blue's own exponent is 0.51.

The two firepower constants are the ones exp05 gave without any control (0.176 and 0.183).
The R² of the alive equations is low (0.13-0.22): single deaths are jumps.

## 4 · Sparsity against accuracy

The health threshold swept, for both ways of fitting. Terms = health equations only.

In [ ]:
def predict(model, **kw):
    return model.simulate(B["X0"], B["shape0"], B["U"], **kw)


sweep = []
for reach_true in [False, True]:
    for thr in [0.02, 0.05, 0.1, 0.15, 0.2, 0.3]:
        m = IntentModel(P).fit(rows, thr_health=thr, reach_true=reach_true, thr_shape=None)
        e = horizon_errors(predict(m), B)
        sweep.append(
            {
                "fitted on": "measured reach" if reach_true else "closure reach",
                "threshold": thr,
                "health terms": int((m.coef["health_blue"] != 0).sum() + (m.coef["health_red"] != 0).sum()),
                "health 4 s": e["health 4 s"],
                "health 8 s": e["health 8 s"],
            }
        )
sweep = pd.DataFrame(sweep)
fig, ax = plt.subplots(figsize=(6, 3.2))
for name, g in sweep.groupby("fitted on"):
    ax.plot(g["health terms"], g["health 4 s"], "o-", label=name)
ax.set(xlabel="terms in the two health equations", ylabel="health error at 4 s")
ax.legend()
sweep.set_index(["fitted on", "threshold"]).round(4)

**What the numbers say:** from 17 terms down to 3 the error at 4 s rises only from about 0.052
to 0.056. The extra terms (alive counts, hold interactions) buy little; the three-term model is
the one to read. At threshold 0.3 a physical term is lost and the error jumps.

## 5 · Predicting ahead

All on the same validation start points, with the real intents applied over the horizon:

- **persistence** and **last slope:** no model
- **generic SINDYc:** `pysindy` with a degree-2 polynomial library on the state and the controls,
  no physics. Once on [health, distance], once on all five states. This is what SINDYc gives
  "out of the box".
- **no control:** the physics library with the controls removed
- **sparse / rich:** the two models of section 3

In [ ]:
def fit_generic(states, threshold, degree=2):
    # Plain pysindy SINDYc: polynomial library on [x, u], u = hold, withdraw, focus (advance is the baseline)
    model = ps.SINDy(
        optimizer=ps.STLSQ(threshold=threshold, alpha=1e-3),
        feature_library=ps.PolynomialLibrary(degree=degree),
        differentiation_method=ps.SmoothedFiniteDifference(smoother_kws={"window_length": 9, "polyorder": 2}),
    )
    groups = [g for _, g in train.groupby(level=0, sort=False)]
    model.fit(
        [g[states].to_numpy() for g in groups],
        t=DT,
        u=[g[CTRL[1:]].to_numpy() for g in groups],
        feature_names=states + CTRL[1:],
    )
    return model


def predict_generic(model, states):
    cols = [STATE.index(s) for s in states]
    X = B["X0"][:, cols].copy()
    out = np.full((len(X), B["U"].shape[1] + 1, len(STATE)), np.nan)
    out[:, 0, cols] = X
    for n in range(B["U"].shape[1]):
        X = np.clip(X + DT * model.predict(X, u=B["U"][:, n, 1:]), 0, 30)  # clipped: polynomials can run away
        out[:, n + 1, cols] = X
    return out


generic = {}
for label, states in {
    "generic SINDYc, [health, distance]": ["health_blue", "health_red", "distance"],
    "generic SINDYc, all five states": STATE,
}.items():
    best = None
    for thr in [0.01, 0.03, 0.1]:
        g = fit_generic(states, thr)
        e = horizon_errors(predict_generic(g, states), B)
        if best is None or e["health 4 s"] < best[1]["health 4 s"]:
            best = (g, e, thr)
    generic[label] = best
    print(f"{label}: best threshold {best[2]}, {int((best[0].coefficients() != 0).sum())} terms")
generic["generic SINDYc, [health, distance]"][0].print(precision=2)

In [ ]:
steps = B["U"].shape[1]
no_control = IntentModel(P, ctrl=False).fit(rows, thr_shape=None)
results = {
    "persistence": horizon_errors(np.repeat(B["X0"][:, None], steps + 1, 1), B),
    "last slope": horizon_errors(
        np.maximum(B["X0"][:, None] + B["slope"][:, None] * (np.arange(steps + 1) * DT)[None, :, None], 0), B
    ),
    **{label: e for label, (_, e, _) in generic.items()},
    "physics library, no control": horizon_errors(predict(no_control), B),
    "sparse (3 health terms)": horizon_errors(predict(sparse), B),
    "rich (13 health terms)": horizon_errors(predict(rich), B),
}
pd.DataFrame(results).T.round(3)

**What the numbers say:**

- **Control matters:** knowing the intent takes the health error at 4 s from 0.079 to 0.052-0.056,
  and the distance error from 0.73 to 0.49.
- **The physics library does not beat generic SINDYc on accuracy.** The degree-2 polynomial SINDYc
  on all five states gets 0.055 at 4 s, as good as the sparse model, but with 147 terms against
  the sparse model's handful. The physics buys **compactness and readability, not accuracy**. It
  is better on distance (0.49 against 0.52).
- Persistence is poor here (0.22 at 4 s): close battles with switching intents change fast.

## 6 · Which part limits accuracy?

The sparse model again, with one of its parts replaced by the truth over the horizon.

In [ ]:
truth_inputs = {
    "model alone": {},
    "+ real distance": {"distance": B["Y"][:, :, 4]},
    "+ real shape (depth, width)": {"shape": B["Yshape"]},
    "+ real alive": {"alive": B["Y"][:, :, 2:4]},
    "+ real distance, shape and alive": {
        "distance": B["Y"][:, :, 4],
        "shape": B["Yshape"],
        "alive": B["Y"][:, :, 2:4],
    },
    "+ real reach": {"reach": B["Yreach"]},
}
pd.DataFrame({name: horizon_errors(predict(sparse, truth=t), B) for name, t in truth_inputs.items()}).T.round(
    3
)

**What the numbers say:** with the real reach the error at 4 s halves (0.028). With the real
distance, shape *and* alive it only reaches about 0.035-0.04. So the limit is the **reach
closure itself**: who is within reach of whom is not fully determined by distance, depth and
width. That is a statement about the state representation (RQ2), not about the fit.

## 7 · Are the terms stable?

30 refits of the sparse model, each on battles resampled with replacement. For every term: how
often it is kept, and the spread of its coefficient.

In [ ]:
episode_ids = rows.index.get_level_values(0)
order = np.argsort(episode_ids, kind="stable")
bounds = np.searchsorted(episode_ids[order], np.unique(episode_ids), side="left").tolist() + [len(order)]
rng = np.random.default_rng(0)
unique = np.unique(episode_ids)
boot = []
for _ in range(30):
    pick = rng.integers(0, len(unique), len(unique))
    idx = np.concatenate([order[bounds[j] : bounds[j + 1]] for j in pick])
    m = IntentModel(P).fit(rows.iloc[idx], thr_health=0.2, reach_true=True, thr_shape=None)
    rec = {
        (eq, n): c
        for eq in ["health_blue", "health_red", "distance"]
        for n, c in zip(m.names, m.coef[eq], strict=True)
        if c != 0
    }
    rec |= {
        (f"alive_{side} exponent", k): v for side in ["blue", "red"] for k, v in m.p[side].items() if v != 0
    }
    boot.append(rec)
boot = pd.DataFrame(boot)
stability = pd.DataFrame(
    {
        "kept in": boot.notna().mean(),
        "mean": boot.mean(),
        "std": boot.std(),
        "min": boot.min(),
        "max": boot.max(),
    }
)
stability.sort_index().round(3)

**Restricted library.** The same resampling (12 refits) without the control x alive candidates
(`Ab*hold`, `Ab*wd`, ...), so the withdraw effect can only enter through reach (`Rb*wd`).

In [ ]:
rng = np.random.default_rng(1)
kept = []
for _ in range(12):
    pick = rng.integers(0, len(unique), len(unique))
    idx = np.concatenate([order[bounds[j] : bounds[j + 1]] for j in pick])
    m = IntentModel(P, exclude=ALIVE_CONTROL_TERMS).fit(
        rows.iloc[idx], thr_health=0.2, reach_true=True, thr_shape=None
    )
    kept.append(
        {
            (eq, n): c
            for eq in ["health_blue", "health_red"]
            for n, c in zip(m.names, m.coef[eq], strict=True)
            if c != 0
        }
    )
kept = pd.DataFrame(kept)
pd.DataFrame({"kept in": kept.notna().mean(), "mean": kept.mean(), "std": kept.std()}).sort_index().round(3)

**What the numbers say:** the health, distance and alive terms are kept in every refit with a
small spread, except the **withdraw effect on Red's losses**: it enters either as `Rb*wd` (53%) or
as `Ab + Ab*wd` (47%). Two candidates describe the same thing (Blue stops firing while withdrawing),
and the data can't tell them apart. The restricted refit above removes the control x alive
candidates.

## 8 · What is each control term worth?

The sparse model with one control effect switched off at a time, and with the intents ignored
altogether (every step predicted as advance/nearest).

In [ ]:
import copy


def without(model, edit):
    m = copy.deepcopy(model)
    edit(m)
    return m


def drop(eq, name):
    def edit(m):
        m.coef[eq][m.names.index(name)] = 0

    return edit


def same_speed(m):  # every movement closes like advance
    for name in ["close*hold", "close*wd"]:
        m.coef["distance"][m.names.index(name)] = m.coef["distance"][m.names.index("close*adv")]
    m.coef["distance"][m.names.index("flee*wd")] = 0


advance_only = np.zeros_like(B["U"])
advance_only[:, :, 0] = 1
ablations = {
    "full sparse model": horizon_errors(predict(sparse), B),
    "without 'withdrawing units don't fire'": horizon_errors(
        predict(without(sparse, drop("health_red", "Rb*wd"))), B
    ),
    "without 'focus kills sooner'": horizon_errors(
        predict(without(sparse, lambda m: m.p["red"].update(focus=0.0))), B
    ),
    "without movement in the distance law": horizon_errors(predict(without(sparse, same_speed)), B),
    "intents ignored (always advance/nearest)": horizon_errors(
        sparse.simulate(B["X0"], B["shape0"], advance_only), B
    ),
}
pd.DataFrame(ablations).T.round(3)

## 9 · How much data does it need?

The sparse coefficients refitted on the first $n$ training battles (the closures stay as fitted
on all 6,000, so this isolates the sparse regression).

In [ ]:
first = np.unique(episode_ids)
data_rows = {}
for n in [30, 100, 300, 1000, 3000, 6000]:
    m = IntentModel(P).fit(rows[episode_ids.isin(first[:n])], thr_health=0.2, reach_true=True, thr_shape=None)
    data_rows[n] = {
        **horizon_errors(predict(m), B)[["health 4 s", "distance 4 s"]],
        "k_blue": -m.coef["health_blue"][m.names.index("Rr")],
        "k_red": -m.coef["health_red"][m.names.index("Rb")],
        "withdraw": m.coef["health_red"][m.names.index("Rb*wd")],
        "terms": m.n_terms(),
    }
pd.DataFrame(data_rows).T.rename_axis("battles").round(3)

## 10 · Does the derivative method matter?

SINDy regresses on derivatives, so how they are computed matters. The sparse model fitted on
smoothed derivatives with windows 5, 9 and 17, and on plain forward differences
$(x_{k+1} - x_k)/\Delta t$, which is the **discrete-time** form $x_{k+1} = x_k + \Delta t\,f(x_k, u_k)$
the planner actually steps with.

In [ ]:
derivative_rows = {}
for label, kw in {
    "smoothed, window 5": {"window": 5},
    "smoothed, window 9": {"window": 9},
    "smoothed, window 17": {"window": 17},
}.items():
    r = add_rates(train[[c for c in train.columns if not c.startswith("d_")]], **kw)
    r = r[(r["alive_blue"] > 0) & (r["alive_red"] > 0)]
    m = IntentModel(P).fit(r, thr_health=0.2, reach_true=True, thr_shape=None)
    derivative_rows[label] = {
        **horizon_errors(predict(m), B)[["health 1 s", "health 4 s", "distance 4 s"]],
        "k_blue": -m.coef["health_blue"][m.names.index("Rr")],
        "k_red": -m.coef["health_red"][m.names.index("Rb")],
        "terms": m.n_terms(),
    }
forward = train.copy()
for c in STATE + SHAPE:
    forward["d_" + c] = (forward.groupby(level=0)[c].shift(-1) - forward[c]) / DT
forward = forward.dropna(subset=["d_health_blue"])
forward = forward[(forward["alive_blue"] > 0) & (forward["alive_red"] > 0)]
m = IntentModel(P).fit(forward, thr_health=0.2, reach_true=True, thr_shape=None)
derivative_rows["forward difference (discrete time)"] = {
    **horizon_errors(predict(m), B)[["health 1 s", "health 4 s", "distance 4 s"]],
    "k_blue": -m.coef["health_blue"][m.names.index("Rr")],
    "k_red": -m.coef["health_red"][m.names.index("Rb")],
    "terms": m.n_terms(),
}
pd.DataFrame(derivative_rows).T.round(3)

## 11 · Formation shape as a state

Depth and width are held at their start values above. Notebook 1 showed that movement changes
them (a marching side stretches into a column; a withdrawing Blue spreads out). Sparse equations
for the four shape variables: a drift and a relaxation term per zone (far / contact) and movement.

In [ ]:
full = IntentModel(P).fit(rows, thr_health=0.2, reach_true=True)
full.print()
pd.DataFrame(
    {
        "shape held": horizon_errors(predict(full), B),
        "shape from its equations": horizon_errors(predict(full, shape="model"), B),
        "real shape": horizon_errors(predict(full, truth={"shape": B["Yshape"]}), B),
        "reach starts from the measured reach": horizon_errors(predict(full, reach0=B["reach0"]), B),
    }
).T.round(3)

In [ ]:
for name in ["sparse", "rich"]:
    m = IntentModel(P).fit(
        rows, **({"thr_health": 0.2, "reach_true": True} if name == "sparse" else {"thr_health": 0.05})
    )
    pickle.dump(m, open(PROCESSED / f"model_{name}.pkl", "wb"))  # used by the planner in notebook 3

**What the numbers say:** the shape equations explain almost nothing step by step (R² below
0.1): formations change through events the team state doesn't see. Modelling shape doesn't
improve the trajectory errors, and even the *real* future shape doesn't. Notebook 3 checks
whether it matters for *decisions*.

## Summary

- SINDYc on a physics-informed library gives a compact controlled model: Lanchester-type fire
  through units in reach, a closing-speed law per movement, and two control effects with clear
  meaning (withdrawing units don't fire; focus fire kills sooner).
- The intents are worth about a third of the health error at 4 s.
- Accuracy is limited by the reach closure, i.e. by what the team-level state can know.

## Questions for you

1. The health equations were cleanest when fitted on the *measured* reach, but the model
   predicts with the *closure's* reach. Is that a fair way to fit? What does each choice optimise?
2. The gates' shapes come from `curve_fit`, the coefficients from STLSQ. Where would you draw
   the line between "SINDy found this" and "I built this in"?
3. Section 7: which terms would you be willing to call a law, and why?
4. Section 6 says the closure is the limit. What would a Level B state (groups) need to contain
   to do better?